In [1]:
language = "english"

dump_file = f"/mnt/Velocity_Vault/Wiki_Dump/Dump/{language}_dump.xml"
raw_corpus_file = f"/mnt/Velocity_Vault/Wiki_Dump/Corpus/{language}_corpus.txt"
corpus_file = f"/mnt/Velocity_Vault/Wiki_Dump/Corpus/{language}_processed.txt"

co_occur_matrix_file = f"/mnt/Velocity_Vault/Wiki_Dump/Memory/{language}_matrix.npy"
pearson_matrix_file = f"/mnt/Velocity_Vault/Wiki_Dump/Memory/{language}_pearson_matrix.npy"
svd_matrix_file = f"/mnt/Velocity_Vault/Wiki_Dump/Memory/{language}_svd_matrix.npy"

In [2]:
import os
import psutil
from tabulate import tabulate

def display_system_info():
    """Display system information using tabulate"""
    info = [
        ["CPU Cores", os.cpu_count()],
        ["CPU Speed (GHz)", f"{(psutil.cpu_freq().max)/1000:.2f}"],
        ["Total Memory (GB)", f"{psutil.virtual_memory().total / 1e9:.2f}"],
        ["Available Memory (GB)", f"{psutil.virtual_memory().available / 1e9:.2f}"]
    ]
    print(tabulate(info, headers=["Metric", "Value"], tablefmt="grid"))

display_system_info()

+-----------------------+---------+
| Metric                |   Value |
+=======================+=========+
| CPU Cores             |   15    |
+-----------------------+---------+
| CPU Speed (GHz)       |    3.94 |
+-----------------------+---------+
| Total Memory (GB)     |   24.87 |
+-----------------------+---------+
| Available Memory (GB) |   16.64 |
+-----------------------+---------+


In [3]:
from collections import Counter

def get_vocabulary(filename, min_freq):
    """Reads corpus file, counts word occurrences.
    Returns a list of words sorted by frequency (highest first)."""
    with open(filename, 'r') as file:
        words = file.read().strip().split()
        
    freq_dict = Counter(words)
    
    word_freq_pairs = [(word, freq) for word, freq in freq_dict.items() if freq >= min_freq]
    
    sorted_word_freq_pairs = sorted(word_freq_pairs, key=lambda x: x[1], reverse=True)
    
    vocab_list = [word for word, _ in sorted_word_freq_pairs]
    
    return vocab_list

vocabulary = get_vocabulary(corpus_file,250)
print("Vocabulary Size - ",len(vocabulary))


Vocabulary Size -  22590


In [4]:
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed
import os

def create_cooccurrence_matrix(corpus_file, matrix_file, window_size, vocab):
    """Create co-occurrence matrix in parallel using position indexing"""
    with open(corpus_file, 'r') as f:
        all_words = f.read().strip().split()
        
    vocab_set = set(vocab)

    for i,word in enumerate(all_words):
        if word not in vocab_set:
            all_words[i] = ""
        
    print("Creating Word Pos...")
    
    word_pos = {}
    for idx in tqdm(range(len(all_words))):
        word = all_words[idx]
        if word != "":
            if word not in word_pos:
                word_pos[word] = []
            word_pos[word].append(idx)
    
    k = len(vocab)
    vocab_idx = {word: i for i, word in enumerate(vocab)}
    
    def compute_row(word, all_words, vocab_idx, window_size, k):
        row = np.zeros(k, dtype=np.int32)
        if word not in word_pos:
            return row
        
        for pos in word_pos[word]:
            start = max(0, pos - window_size)
            end = min(len(all_words), pos + window_size + 1)
            
            for j in range(start, end):
                if j != pos and all_words[j] != "":
                    context_word = all_words[j]
                    distance = abs(j - pos)
                    row[vocab_idx[context_word]] += window_size - distance + 1
        
        return row
    
    print("Computing co-occurrence matrix")
    rows = Parallel(n_jobs=os.cpu_count(), require='sharedmem')(
        delayed(compute_row)(word, all_words, vocab_idx, window_size, k)
        for word in tqdm(vocab, desc="Processing words")
    )
    
    matrix = np.array(rows, dtype=np.int32)
    
    print("Saving matrix to file...")
    np.save(matrix_file, matrix)
    
    print(f"Co-occurrence matrix saved to {matrix_file}")

create_cooccurrence_matrix(corpus_file, co_occur_matrix_file, 4, vocabulary)

Creating Word Pos...


100%|██████████| 113146814/113146814 [00:17<00:00, 6309944.88it/s]


Computing co-occurrence matrix


Processing words: 100%|██████████| 22590/22590 [04:49<00:00, 78.07it/s]  


Saving matrix to file...
Co-occurrence matrix saved to /mnt/Velocity_Vault/Wiki_Dump/Memory/english_matrix.npy
